# imports  & function definitions | 1

In [ ]:
import os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RANSACRegressor

# optional display/plot helpers
from IPython.display import display
import matplotlib.pyplot as plt


## ---- robust k-NN filter


In [ ]:
def robust_knn_filter(points, k_knn=8, mad_factor=3.0):
    n = points.shape[0]
    k = min(k_knn, max(1, n-1))
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='kd_tree').fit(points)
    dists, _ = nbrs.kneighbors(points)
    mean_d = dists[:,1:].mean(axis=1)
    median_md = np.median(mean_d)
    mad = np.median(np.abs(mean_d - median_md)) + 1e-12
    cutoff = median_md + mad_factor * mad
    mask = mean_d <= cutoff
    return mask, {'median_md': float(median_md), 'mad': float(mad), 'cutoff': float(cutoff)}


## ---- density filter (d_p = sum 1/||p-q||^2)


In [ ]:


def density_filter(points, mask_prev, eps_scale=1.0, remove_quantile=0.05):
    idx = np.nonzero(mask_prev)[0]
    pts = points[idx]
    if pts.shape[0] == 0:
        return np.zeros_like(mask_prev, dtype=bool), {}
    tree = cKDTree(pts)
    if pts.shape[0] > 1:
        d1, _ = tree.query(pts, k=2)
        avg_nn = float(d1[:,1].mean())
    else:
        avg_nn = 0.0
    radius = max(1e-9, eps_scale * avg_nn * 2.0)
    neighbors = tree.query_ball_point(pts, r=radius)
    densities = np.zeros(pts.shape[0], dtype=float)
    for i in range(pts.shape[0]):
        neigh_idx = [j for j in neighbors[i] if j != i]
        if len(neigh_idx) == 0:
            densities[i] = 0.0
        else:
            dists = np.linalg.norm(pts[i] - pts[neigh_idx], axis=1)
            densities[i] = np.sum(1.0 / (dists**2 + 1e-12))
    cut = float(np.quantile(densities, remove_quantile))
    mask_keep = densities > cut
    full_mask = np.zeros_like(mask_prev, dtype=bool)
    full_mask[idx[mask_keep]] = True
    return full_mask, {'avg_nn': avg_nn, 'radius': radius, 'density_cut': cut}


## ---- LOF filter


In [ ]:

def lof_filter(points, mask_prev, lof_k=20, lof_fraction=0.03):
    idx = np.nonzero(mask_prev)[0]
    pts = points[idx]
    if pts.shape[0] < 3:
        return mask_prev.copy(), {}
    n_neighbors = min(lof_k, max(2, pts.shape[0]-1))
    lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=False)
    _ = lof.fit_predict(pts)
    lof_scores = -lof.negative_outlier_factor_
    q = min(0.5, max(0.0, lof_fraction))
    if q > 0:
        thresh = float(np.quantile(lof_scores, 1.0 - q))
    else:
        thresh = lof_scores.max() + 1.0
    keep_local = lof_scores <= thresh
    full_mask = np.zeros_like(mask_prev, dtype=bool)
    full_mask[idx[keep_local]] = True
    return full_mask, {'n_neighbors': n_neighbors, 'thresh': thresh, 'kept': int(keep_local.sum())}


## ---- adaptive DBSCAN


In [ ]:

def adaptive_dbscan(points, mask_prev, db_min_samples=5, k_dist_for_eps=4, eps_percentile=90, eps_relax_factor=1.5, eps_relax_steps=6):
    idx = np.nonzero(mask_prev)[0]
    pts = points[idx]
    if pts.shape[0] < db_min_samples:
        return mask_prev.copy(), {'note': 'not_enough_points'}
    scaler = StandardScaler().fit(pts)
    scaled = scaler.transform(pts)
    kd = min(k_dist_for_eps, max(1, scaled.shape[0]-1))
    nbrs = NearestNeighbors(n_neighbors=kd+1).fit(scaled)
    dists, _ = nbrs.kneighbors(scaled)
    kdist = dists[:, kd]
    eps0 = float(np.percentile(kdist, eps_percentile))
    if eps0 <= 0:
        eps0 = 1e-6
    eps = eps0
    chosen = None
    n_clusters = 0
    for step in range(eps_relax_steps):
        db = DBSCAN(eps=eps, min_samples=db_min_samples).fit(scaled)
        labels = db.labels_
        kept_count = int((labels != -1).sum())
        n_clusters = len(set(labels)) - ( -1 in labels )
        if kept_count > 0 and n_clusters > 0:
            chosen = (labels != -1)
            break
        eps *= eps_relax_factor
    if chosen is None:
        full_mask = np.zeros_like(mask_prev, dtype=bool)
        full_mask[idx] = True
        return full_mask, {'note': 'dbscan_failed', 'eps_tried': eps}
    full_mask = np.zeros_like(mask_prev, dtype=bool)
    full_mask[idx[chosen]] = True
    return full_mask, {'eps_used': eps, 'clusters': n_clusters, 'kept': int(chosen.sum())}


## ---- preserve_curve_points (PCA + RANSAC poly3 on two projections)


In [ ]:

def preserve_curve_points(points, current_mask, *, min_cluster_size=6, cluster_eps=None, cluster_min_samples=5,
                          curve_tol_factor=3.0, curve_tol_abs=None, snap_to_curve=False, n_curve_samples=200,
                          ransac_residual=0.02, mfe_threshold=0.02, verbose=False):
    N = points.shape[0]
    idx_surv = np.nonzero(current_mask)[0]
    pts_surv = points[idx_surv]
    if pts_surv.shape[0] < min_cluster_size:
        if verbose: print("preserve_curve_points: not enough survived points.")
        return current_mask.copy(), None
    if cluster_eps is None:
        tree = cKDTree(pts_surv)
        if pts_surv.shape[0] > 1:
            d1, _ = tree.query(pts_surv, k=2)
            median_nn = float(np.median(d1[:,1]))
            eps0 = max(1e-6, 2.0 * median_nn)
        else:
            eps0 = 1e-6
    else:
        eps0 = cluster_eps
    db = DBSCAN(eps=eps0, min_samples=cluster_min_samples).fit(pts_surv)
    labels = db.labels_
    unique_labels = [lab for lab in set(labels) if lab != -1]
    if len(unique_labels) == 0:
        if verbose: print("preserve_curve_points: no clusters found.")
        return current_mask.copy(), None
    tree_all = cKDTree(points)
    if points.shape[0] > 1:
        d_all, _ = tree_all.query(points, k=2)
        avg_nn = float(np.mean(d_all[:,1]))
    else:
        avg_nn = 0.0
    curve_tol = curve_tol_abs if curve_tol_abs is not None else max(1e-9, curve_tol_factor * avg_nn)
    if verbose:
        print(f"preserve_curve_points: curve_tol={curve_tol:.6g} (avg_nn={avg_nn:.6g})")
    all_curve_samples = []
    def fit_cluster_get_samples(cluster_pts):
        pca = PCA(n_components=3)
        pts_r = pca.fit_transform(cluster_pts)
        proj1 = np.column_stack([pts_r[:,0], pts_r[:,1]])
        proj2 = np.column_stack([pts_r[:,0], pts_r[:,2]])
        def normalize_proj(proj):
            mins = proj.min(axis=0)
            maxs = proj.max(axis=0)
            scale = (maxs - mins)
            scale[scale == 0] = 1.0
            proj_n = (proj - mins) / scale
            proj_n = proj_n - 0.5
            return proj_n, mins, scale
        p1n, mins1, scale1 = normalize_proj(proj1)
        p2n, mins2, scale2 = normalize_proj(proj2)
        def ransac_cubic(x, y):
            Xpoly = PolynomialFeatures(degree=3, include_bias=True).fit_transform(x.reshape(-1,1))
            base = LinearRegression()
            # Try to construct RANSACRegressor using the modern 'estimator' param;
            # if that raises TypeError (older/newsklearn), fall back to 'base_estimator'.
            try:
                ransac = RANSACRegressor(estimator=base, residual_threshold=ransac_residual, random_state=0)
            except TypeError:
                # older sklearn versions use base_estimator keyword
                ransac = RANSACRegressor(base_estimator=base, residual_threshold=ransac_residual, random_state=0)
            try:
                ransac.fit(Xpoly, y)
            except Exception:
                return None, None
            inlier_mask = getattr(ransac, 'inlier_mask_', None)
            return ransac, inlier_mask

        r1, m1 = ransac_cubic(p1n[:,0], p1n[:,1])
        r2, m2 = ransac_cubic(p2n[:,0], p2n[:,1])
        def compute_mfe(ransac, proj_n):
            if ransac is None:
                return np.inf
            xs = proj_n[:,0]
            xmin, xmax = xs.min(), xs.max()
            ts = np.linspace(xmin, xmax, max(50, len(xs)))
            Xs_poly = PolynomialFeatures(degree=3, include_bias=True).fit_transform(ts.reshape(-1,1))
            try:
                ys = ransac.predict(Xs_poly)
            except Exception:
                return np.inf
            samples = np.column_stack([ts, ys])
            kd = cKDTree(samples)
            dists, _ = kd.query(proj_n)
            return float(dists.mean())
        mfe1 = compute_mfe(r1, p1n) if r1 is not None else np.inf
        mfe2 = compute_mfe(r2, p2n) if r2 is not None else np.inf
        if min(mfe1, mfe2) > mfe_threshold:
            return None
        if mfe1 <= mfe2:
            r = r1; proj_n = p1n; mins = mins1; scale = scale1
            xs = proj_n[:,0]; xmin, xmax = xs.min(), xs.max()
            ts = np.linspace(xmin, xmax, n_curve_samples)
            Xs_poly = PolynomialFeatures(degree=3, include_bias=True).fit_transform(ts.reshape(-1,1))
            ys = r.predict(Xs_poly)
            samples_proj = np.column_stack([ts, ys])
            samples_proj_den = (samples_proj + 0.5) * scale + mins
            try:
                polyX = PolynomialFeatures(degree=3, include_bias=True).fit_transform(pts_r[:,0].reshape(-1,1))
                linZ = LinearRegression().fit(polyX, pts_r[:,2])
                Zp = linZ.predict(PolynomialFeatures(degree=3, include_bias=True).fit_transform((samples_proj_den[:,0]).reshape(-1,1)))
            except Exception:
                from scipy.interpolate import interp1d
                fz = interp1d(pts_r[:,0], pts_r[:,2], bounds_error=False, fill_value="extrapolate")
                Zp = fz(samples_proj_den[:,0])
            samples_pca = np.column_stack([samples_proj_den[:,0], samples_proj_den[:,1], Zp])
            samples_world = pca.inverse_transform(samples_pca)
            return samples_world
        else:
            r = r2; proj_n = p2n; mins = mins2; scale = scale2
            xs = proj_n[:,0]; xmin, xmax = xs.min(), xs.max()
            ts = np.linspace(xmin, xmax, n_curve_samples)
            Xs_poly = PolynomialFeatures(degree=3, include_bias=True).fit_transform(ts.reshape(-1,1))
            ys = r.predict(Xs_poly)
            samples_proj = np.column_stack([ts, ys])
            samples_proj_den = (samples_proj + 0.5) * scale + mins
            try:
                polyX = PolynomialFeatures(degree=3, include_bias=True).fit_transform(pts_r[:,0].reshape(-1,1))
                linY = LinearRegression().fit(polyX, pts_r[:,1])
                Yp = linY.predict(PolynomialFeatures(degree=3, include_bias=True).fit_transform((samples_proj_den[:,0]).reshape(-1,1)))
            except Exception:
                from scipy.interpolate import interp1d
                fy = interp1d(pts_r[:,0], pts_r[:,1], bounds_error=False, fill_value="extrapolate")
                Yp = fy(samples_proj_den[:,0])
            samples_pca = np.column_stack([samples_proj_den[:,0], Yp, samples_proj_den[:,1]])
            samples_world = pca.inverse_transform(samples_pca)
            return samples_world
    for lab in unique_labels:
        ids = np.where(labels == lab)[0]
        if len(ids) < min_cluster_size:
            continue
        cluster_pts = pts_surv[ids]
        samples = fit_cluster_get_samples(cluster_pts)
        if samples is not None and samples.shape[0] > 0:
            all_curve_samples.append(samples)
    if len(all_curve_samples) == 0:
        if verbose: print("preserve_curve_points: no curve samples produced.")
        return current_mask.copy(), None
    all_samples = np.vstack(all_curve_samples)
    kd_all = cKDTree(all_samples)
    dists_to_curve, idx_near = kd_all.query(points)
    preserve_mask = dists_to_curve <= curve_tol
    new_mask = current_mask | preserve_mask
    snapped_points = None
    if snap_to_curve:
        snapped_points = points.copy()
        idx_to_snap = np.where((~current_mask) & preserve_mask)[0]
        if idx_to_snap.size > 0:
            _, nearest_idxs = kd_all.query(points[idx_to_snap])
            snapped_points[idx_to_snap] = all_samples[nearest_idxs]
    return new_mask, snapped_points

#  2 -- Set input path and parameters, run pipeline 


    ## input :
    ## params : we can change :)

In [ ]:
# input_path = '1.csv'  

# # load and check
# df = pd.read_csv(input_path, sep=r'\s+|,|\t', engine='python')

# col_map = {}
# for c in df.columns:
#     if c.lower() == 'x': col_map[c] = 'X'
#     if c.lower() == 'y': col_map[c] = 'Y'
#     if c.lower() == 'z': col_map[c] = 'Z'
# df = df.rename(columns=col_map)
# if not all(k in df.columns for k in ['X','Y','Z']):
#     raise ValueError("Input CSV must contain X,Y,Z columns (case-insensitive). Found: " + ", ".join(df.columns))

# points = df[['X','Y','Z']].to_numpy()
# N = points.shape[0]
# print(f"Loaded {N} points from {input_path}")




# params = {
#  'k_knn': 20, 'mad_factor': 8.0,
#  'eps_scale': 1.6, 'density_remove_quantile': 0.03,
#  'lof_k': 30, 'lof_fraction': 0.01,
#  'db_min_samples': 3, 'k_dist_for_eps': 4, 'eps_percentile': 90,
#  'eps_relax_factor': 2.0, 'eps_relax_steps': 8,
#  'preserve_curves': False,
#  'curve_min_cluster': 6, 'cluster_eps': None, 'cluster_min_samples': 4,
#  'curve_tol_factor': 2.0, 'curve_tol_abs': None, 'snap_to_curve': False,
#  'n_curve_samples': 200, 'ransac_residual': 0.03, 'mfe_threshold': 0.03,
#  'verbose': False 
# }


# # params = {

# #  'k_knn': 20, 'mad_factor': 8.0,
# #  'eps_scale': 1.6, 'density_remove_quantile': 0.03,
# #  'lof_k': 30, 'lof_fraction': 0.01,
# #  'db_min_samples': 3, 'k_dist_for_eps': 4, 'eps_percentile': 90,
# #  'eps_relax_factor': 2.0, 'eps_relax_steps': 8,
# #  # === change here: enable curve preservation and soften curve tolerance slightly ===
# #  'preserve_curves': True,
# #  'curve_min_cluster': 5,        # allow smaller clusters to be considered
# #  'cluster_eps': None,           # None => auto (usually fine); override manually if needed
# #  'cluster_min_samples': 4,
# #  'curve_tol_factor': 3.5,       # from 2.0 -> 3.5 (looser: retains points slightly farther from curve)
# #  'curve_tol_abs': None,
# #  'snap_to_curve': False,        # False: keep-only; coordinates unchanged
# #  'n_curve_samples': 300,
# #  'ransac_residual': 0.03,
# #  'mfe_threshold': 0.04,
# #  'verbose': True
# # } 

# mask_knn, diag_knn = robust_knn_filter(points, k_knn=params['k_knn'], mad_factor=params['mad_factor'])
# print(f"After robust k-NN: {mask_knn.sum()}/{N}")

# mask_density, diag_density = density_filter(points, mask_knn, eps_scale=params['eps_scale'], remove_quantile=params['density_remove_quantile'])
# print(f"After density filter: {mask_density.sum()}/{N}")

# mask_combined = mask_density.copy()

# mask_lof, diag_lof = lof_filter(points, mask_combined, lof_k=params['lof_k'], lof_fraction=params['lof_fraction'])
# print(f"After LOF: {mask_lof.sum()}/{N}")

# mask_db, diag_db = adaptive_dbscan(points, mask_lof,
#                                    db_min_samples=params['db_min_samples'],
#                                    k_dist_for_eps=params['k_dist_for_eps'],
#                                    eps_percentile=params['eps_percentile'],
#                                    eps_relax_factor=params['eps_relax_factor'],
#                                    eps_relax_steps=params['eps_relax_steps'])
# if mask_db.sum() == 0:
#     print("DBSCAN removed all points; fallback to LOF-filtered set.")
#     mask_final = mask_lof.copy()
# else:
#     mask_final = mask_db.copy()
# print(f"After DBSCAN/fallback: {mask_final.sum()}/{N}")


# # save intermediate
# out_dir = os.path.dirname(input_path) or '.'
# out_clean_tmp = os.path.join(out_dir, "1_cleaned_refined.csv")
# pd.DataFrame(points[mask_final], columns=['X','Y','Z']).to_csv(out_clean_tmp, index=False)
# df_out = df.copy()
# df_out['kept_after_knn'] = mask_knn.astype(int)
# df_out['kept_after_density'] = mask_density.astype(int)
# df_out['kept_after_lof'] = mask_lof.astype(int)
# df_out['kept_after_dbscan'] = mask_db.astype(int)
# df_out['kept_final'] = mask_final.astype(int)
# out_annot_tmp = os.path.join(out_dir, "1_annotated_refined.csv")
# df_out.to_csv(out_annot_tmp, index=False)
# print(f"Saved intermediate outputs: {out_clean_tmp}, {out_annot_tmp}")

# # preserve curves
# if params['preserve_curves']:
#     new_mask, snapped = preserve_curve_points(points, mask_final,
#                                               min_cluster_size=params['curve_min_cluster'],
#                                               cluster_eps=params['cluster_eps'],
#                                               cluster_min_samples=params['cluster_min_samples'],
#                                               curve_tol_factor=params['curve_tol_factor'],
#                                               curve_tol_abs=params['curve_tol_abs'],
#                                               snap_to_curve=params['snap_to_curve'],
#                                               n_curve_samples=params['n_curve_samples'],
#                                               ransac_residual=params['ransac_residual'],
#                                               mfe_threshold=params['mfe_threshold'],
#                                               verbose=params['verbose'])
#     if new_mask is None:
#         print("No curves detected; keeping previous final mask.")
#         new_mask = mask_final.copy()
#     out_clean_curve = os.path.join(out_dir, "1_cleaned_with_curve.csv")
#     if params['snap_to_curve'] and snapped is not None:
#         pd.DataFrame(snapped[new_mask], columns=['X','Y','Z']).to_csv(out_clean_curve, index=False)
#     else:
#         pd.DataFrame(points[new_mask], columns=['X','Y','Z']).to_csv(out_clean_curve, index=False)
#     df_curve = df.copy()
#     df_curve['kept_final_before_curve'] = mask_final.astype(int)
#     df_curve['kept_after_curve_preserve'] = new_mask.astype(int)
#     out_annot_curve = os.path.join(out_dir, "1_annotated_with_curve.csv")
#     df_curve.to_csv(out_annot_curve, index=False)
#     print(f"Saved curve outputs: {out_clean_curve}, {out_annot_curve}")
# else:
#     new_mask = mask_final.copy()


# =========================
# Cell 2 (revised): run and save pipeline with stronger curve protection
# =========================
# =========================
# Cell 2 (advanced): run pipeline with advanced curve protection
# =========================

import os
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# Input file path
input_path = "./1.csv"

# Verify base functions are defined
required_funcs = ['robust_knn_filter', 'density_filter', 'lof_filter', 'adaptive_dbscan', 'preserve_curve_points']
missing = [f for f in required_funcs if f not in globals()]
if len(missing) > 0:
    raise RuntimeError(f"base functions not defined in environment: {missing}")

# Load data
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Input file {input_path} not found.")
df = pd.read_csv(input_path, sep=r'\s+|,|\t', engine='python')

# normalize columns
col_map = {}
for c in df.columns:
    if c.lower() == 'x': col_map[c] = 'X'
    if c.lower() == 'y': col_map[c] = 'Y' 
    if c.lower() == 'z': col_map[c] = 'Z'
df = df.rename(columns=col_map)

points = df[['X','Y','Z']].to_numpy()
N = points.shape[0]
print(f"Loaded {N} points from {input_path}")

# -----------------------
# Advanced parameters for curve preservation
# -----------------------
params = {
    # Initial filters - relaxed to preserve curve structures
    'k_knn': 10, 
    'mad_factor': 4.0,                # less aggressive
    
    'eps_scale': 1.2, 
    'density_remove_quantile': 0.01,  # remove only the least-dense points
    
    'lof_k': 15, 
    'lof_fraction': 0.005,            # very conservative
    
    'db_min_samples': 3, 
    'k_dist_for_eps': 3, 
    'eps_percentile': 85,
    'eps_relax_factor': 1.8, 
    'eps_relax_steps': 10,            # more chances to find clusters
    
    # === Advanced curve protection ===
    'preserve_curves': True,
    'curve_min_cluster': 3,           # even very small clusters
    'cluster_eps': None,              
    'cluster_min_samples': 3,
    'curve_tol_factor': 4.0,          # very loose - retain points far from curve too
    'curve_tol_abs': None,           
    'snap_to_curve': False,          
    'n_curve_samples': 500,           # very dense sampling for full coverage
    'ransac_residual': 0.05,          # very lenient
    'mfe_threshold': 0.08,            # looser fitting error allowed
    'verbose': True,
    
    # New parameters for bridge point preservation between curves
    'bridge_points_enabled': True,    # enable bridge point preservation
    'bridge_max_distance': 2.5,       # max distance to connect curves
    'min_bridge_points': 2,           # min points to form a bridge
    'curve_extension_factor': 1.2,    # curve extension for better coverage
}

# -----------------------
# New function: detect and preserve bridge points between curves
# -----------------------
def preserve_bridge_points(points, curves, max_distance=2.5, min_points=2):
    """Preserve points located between different curves (bridge points)."""
    if len(curves) < 2:
        return set()
    
    bridge_points = set()
    
    # For each pair of curves
    for i in range(len(curves)):
        for j in range(i + 1, len(curves)):
            curve1 = curves[i]
            curve2 = curves[j]
            
            # Curve endpoint points
            end1 = curve1['points'][-1]
            start1 = curve1['points'][0]
            end2 = curve2['points'][-1] 
            start2 = curve2['points'][0]
            
            # Compute distance between curve endpoints
            distances = [
                np.linalg.norm(end1 - start2),
                np.linalg.norm(end1 - end2),
                np.linalg.norm(start1 - start2),
                np.linalg.norm(start1 - end2)
            ]
            
            min_dist = min(distances)
            
            if min_dist <= max_distance:
                # Find points located between these two curves
                mid_point = None
                if min_dist == distances[0]:
                    mid_point = (end1 + start2) / 2
                elif min_dist == distances[1]:
                    mid_point = (end1 + end2) / 2
                elif min_dist == distances[2]:
                    mid_point = (start1 + start2) / 2
                else:
                    mid_point = (start1 + end2) / 2
                
                # Find points nearest to the midpoint
                tree = cKDTree(points)
                dists, indices = tree.query(mid_point, k=min(10, len(points)))
                
                # Add nearby points to the bridge set
                for idx, dist in zip(indices, dists):
                    if dist <= max_distance * 1.5:
                        bridge_points.add(idx)
    
    return bridge_points

# -----------------------
# New function: extend curves to cover adjacent points
# -----------------------
def extend_curves_to_neighbors(points, curves, extension_factor=1.2):
    """Extend curves to cover adjacent points."""
    extended_curves = []
    
    for curve in curves:
        if len(curve['points']) < 3:
            continue
            
        # Compute curve direction at endpoints
        start_dir = curve['points'][1] - curve['points'][0]
        end_dir = curve['points'][-1] - curve['points'][-2]
        
        # Normalize direction vectors
        start_dir_norm = start_dir / (np.linalg.norm(start_dir) + 1e-8)
        end_dir_norm = end_dir / (np.linalg.norm(end_dir) + 1e-8)
        
        # Average curve length
        curve_length = np.sum(np.linalg.norm(np.diff(curve['points'], axis=0), axis=1))
        extension_length = curve_length * extension_factor / len(curve['points'])
        
        # Extended endpoint positions
        extended_start = curve['points'][0] - start_dir_norm * extension_length
        extended_end = curve['points'][-1] + end_dir_norm * extension_length
        
        # Build extended curve
        extended_points = np.vstack([extended_start, curve['points'], extended_end])
        
        extended_curve = curve.copy()
        extended_curve['points'] = extended_points
        extended_curves.append(extended_curve)
    
    return extended_curves

# -----------------------
# 1) Robust k-NN filter - relaxed
# -----------------------
mask_knn, diag_knn = robust_knn_filter(points, k_knn=params['k_knn'], mad_factor=params['mad_factor'])
print(f"After robust k-NN: {mask_knn.sum()}/{N}")

# -----------------------
# 2) Density filter - remove only the least-dense points
# -----------------------
mask_density, diag_density = density_filter(points, mask_knn, eps_scale=params['eps_scale'], remove_quantile=params['density_remove_quantile'])
print(f"After density filter: {mask_density.sum()}/{N}")

# -----------------------
# 3) LOF filter - very conservative
# -----------------------
mask_combined = mask_density.copy()
mask_lof, diag_lof = lof_filter(points, mask_combined, lof_k=params['lof_k'], lof_fraction=params['lof_fraction'])
print(f"After LOF: {mask_lof.sum()}/{N}")

# -----------------------
# 4) Adaptive DBSCAN - higher sensitivity
# -----------------------
mask_db, diag_db = adaptive_dbscan(points, mask_lof,
                                   db_min_samples=params['db_min_samples'],
                                   k_dist_for_eps=params['k_dist_for_eps'],
                                   eps_percentile=params['eps_percentile'],
                                   eps_relax_factor=params['eps_relax_factor'],
                                   eps_relax_steps=params['eps_relax_steps'])

if mask_db.sum() == 0:
    print("DBSCAN removed all points; fallback to LOF-filtered set.")
    mask_final = mask_lof.copy()
else:
    mask_final = mask_db.copy()
print(f"After DBSCAN/fallback: {mask_final.sum()}/{N}")

# Intermediate save
out_dir = os.path.dirname(input_path) or '.'
out_clean_tmp = os.path.join(out_dir, "1_cleaned_refined.csv")
pd.DataFrame(points[mask_final], columns=['X','Y','Z']).to_csv(out_clean_tmp, index=False)

df_out = df.copy()
df_out['kept_after_knn'] = mask_knn.astype(int)
df_out['kept_after_density'] = mask_density.astype(int)
df_out['kept_after_lof'] = mask_lof.astype(int)
df_out['kept_after_dbscan'] = mask_db.astype(int)
df_out['kept_final'] = mask_final.astype(int)

out_annot_tmp = os.path.join(out_dir, "1_annotated_refined.csv")
df_out.to_csv(out_annot_tmp, index=False)
print(f"Saved intermediate outputs")

# -----------------------
# 5) Advanced curve and bridge point protection
# -----------------------
if params['preserve_curves']:
    print("\n--------------------------------\n")
    print("\n=== Starting advanced curve protection ===")
    
    # Stage 1: standard curve protection
    new_mask, snapped = preserve_curve_points(points, mask_final,
                                              min_cluster_size=params['curve_min_cluster'],
                                              cluster_eps=params['cluster_eps'],
                                              cluster_min_samples=params['cluster_min_samples'],
                                              curve_tol_factor=params['curve_tol_factor'],
                                              curve_tol_abs=params['curve_tol_abs'],
                                              snap_to_curve=params['snap_to_curve'],
                                              n_curve_samples=params['n_curve_samples'],
                                              ransac_residual=params['ransac_residual'],
                                              mfe_threshold=params['mfe_threshold'],
                                              verbose=params['verbose'])
    
    if new_mask is None:
        print("No curve preservation applied.")
        new_mask = mask_final.copy()
    
    # Detect curves for Stage 2 (bridge points)
    print("\n--------------------------------\n")
    print("=== Detecting curves for bridge point preservation ===")
    idx_surv = np.nonzero(new_mask)[0]
    pts_surv = points[idx_surv]
    
    curves_detected = []
    if len(pts_surv) >= params['curve_min_cluster']:
        # DBSCAN to detect curve-shaped clusters
        from sklearn.cluster import DBSCAN
        from sklearn.decomposition import PCA
        
        scaler = StandardScaler()
        pts_scaled = scaler.fit_transform(pts_surv)
        
        nbrs = NearestNeighbors(n_neighbors=min(5, len(pts_scaled)-1)).fit(pts_scaled)
        distances, _ = nbrs.kneighbors(pts_scaled)
        k_distances = distances[:, -1]
        eps = np.percentile(k_distances, 80)
        
        db = DBSCAN(eps=eps, min_samples=params['curve_min_cluster']).fit(pts_scaled)
        labels = db.labels_
        
        for label in set(labels):
            if label == -1:
                continue
                
            cluster_indices = np.where(labels == label)[0]
            cluster_points = pts_surv[cluster_indices]
            
            if len(cluster_points) < params['curve_min_cluster']:
                continue
            
            # Check linearity with PCA
            pca = PCA(n_components=3)
            pca.fit(cluster_points)
            explained_variance = pca.explained_variance_ratio_
            
            if explained_variance[0] > 0.7:  # linearity threshold
                # Sort points by first principal component
                projections = pca.transform(cluster_points)[:, 0]
                order = np.argsort(projections)
                ordered_points = cluster_points[order]
                ordered_indices = idx_surv[cluster_indices[order]]
                
                curves_detected.append({
                    'points': ordered_points,
                    'indices': ordered_indices,
                    'pca_ratio': explained_variance[0],
                    'length': len(ordered_points)
                })
    
    print(f"Curves detected: {len(curves_detected)}")
    
    # Stage 2: preserve bridge points between curves
    if params['bridge_points_enabled'] and len(curves_detected) >= 2:
        print("\n--------------------------------\n")
        print("=== Preserving bridge points between curves ===")
        
        # Extend curves
        extended_curves = extend_curves_to_neighbors(points, curves_detected, 
                                                   params['curve_extension_factor'])
        
        # Detect bridge points
        bridge_indices = preserve_bridge_points(points, extended_curves,
                                              max_distance=params['bridge_max_distance'],
                                              min_points=params['min_bridge_points'])
        
        print(f"Bridge points detected: {len(bridge_indices)}")
        
        # Add bridge points to final mask
        for idx in bridge_indices:
            new_mask[idx] = True
    
    # Stage 3: recover points near extended curves extended
    if len(curves_detected) > 0:
        print("\n--------------------------------\n")
        print("=== Recovering points near curves ===")
        
        # Generate extended curve samples
        all_curve_samples = []
        for curve in extended_curves if 'extended_curves' in locals() else curves_detected:
            # Sample from curve
            t = np.linspace(0, 1, params['n_curve_samples'])
            if len(curve['points']) >= 4:
                # Spline interpolation
                from scipy.interpolate import CubicSpline
                try:
                    cs_x = CubicSpline(np.linspace(0, 1, len(curve['points'])), curve['points'][:, 0])
                    cs_y = CubicSpline(np.linspace(0, 1, len(curve['points'])), curve['points'][:, 1])
                    cs_z = CubicSpline(np.linspace(0, 1, len(curve['points'])), curve['points'][:, 2])
                    
                    samples = np.column_stack([cs_x(t), cs_y(t), cs_z(t)])
                    all_curve_samples.append(samples)
                except:
                    # fallback: linear sampling
                    indices = np.linspace(0, len(curve['points'])-1, params['n_curve_samples']).astype(int)
                    samples = curve['points'][indices]
                    all_curve_samples.append(samples)
        
        if all_curve_samples:
            all_samples = np.vstack(all_curve_samples)
            kd_curves = cKDTree(all_samples)
            
            # Find points near curves not yet retained
            not_in_mask = ~new_mask
            if np.any(not_in_mask):
                dists, _ = kd_curves.query(points[not_in_mask])
                close_points = np.where(not_in_mask)[0][dists <= params['bridge_max_distance'] * 1.2]
                
                print(f"Points near curves recovered: {len(close_points)}")
                new_mask[close_points] = True
    
    kept_before = int(mask_final.sum())
    kept_after = int(new_mask.sum())
    print(f"\nCurve protection results:")
    print(f"Before protection: {kept_before}")
    print(f"After protection:  {kept_after}")
    print(f"Points added:      {kept_after - kept_before}")
    
        # final save
    out_clean_curve = os.path.join(out_dir, "1_cleaned_advanced_curve.csv")
    pd.DataFrame(points[new_mask], columns=['X','Y','Z']).to_csv(out_clean_curve, index=False)
    
        # save advanced annotation
    out_annot_curve = os.path.join(out_dir, "1_annotated_advanced_curve.csv")
    df_curve = df.copy()
    df_curve['kept_final_before_curve'] = mask_final.astype(int)
    df_curve['kept_after_curve_preserve'] = new_mask.astype(int)
    
        # Add curve metadata
    if curves_detected:
        df_curve['curve_member'] = 0
        for i, curve in enumerate(curves_detected):
            df_curve.loc[curve['indices'], 'curve_member'] = i + 1
    
    df_curve.to_csv(out_annot_curve, index=False)
    
    print(f"\nOutput files saved:")
    print(f"- Clean point cloud: {out_clean_curve}")
    print(f"- Annotated data: {out_annot_curve}")

else:
    new_mask = mask_final.copy()
    print("Curve protection is disabled")

# -----------------------
# Final summary
# -----------------------
final_count = new_mask.sum()
print("\n==================================\n")
print(f"\n=== Final Summary ===")
print(f"input points: {N}")
print(f"Final points retained: {final_count} ({final_count/N*100:.1f}%)")
print(f"Removed: {N - final_count} ({(N-final_count)/N*100:.1f}%)")

# 4 - Plot results 

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(12,6))
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(points[:,0], points[:,1], points[:,2], s=6)
ax1.set_title('Original')

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(points[new_mask,0], points[new_mask,1], points[new_mask,2], s=12)
ax2.set_title('Cleaned (final, with curve-preserve)')
plt.show()
